# Earth-2 Models on a T4

Measures which Earth2Studio weather models fit on a free Kaggle T4 GPU
(16 GB), how long a 3-day forecast takes, and which inputs of the
Hot-Dry-Windy Index each model outputs.

Run on Kaggle with **Accelerator: GPU T4 x2** and **Internet: on**. The
notebook uses one GPU. Global models start from ERA5 on Google Cloud
(ARCO), which needs no account.

Output: `/kaggle/working/earth2_models_t4.json`.

In [ ]:
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git@b38fcb2799d7dbc146fa60459f3f9823394a8bf1" "earth2studio[data,dlwp,fcn3,sfno]==0.20.0" 2>&1 | tail -3
!pip install -q "earth2studio[atlas,stormcast-conus]==0.20.0" 2>&1 | tail -3

In [ ]:
import gc
import json
import platform
import subprocess
import time
from datetime import datetime
from pathlib import Path

import torch

import earth2studio

environment = {
    "gpu": torch.cuda.get_device_name(0),
    "gpu_memory_gb": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1),
    "driver": subprocess.run(
        ["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    ).stdout.strip().splitlines()[0],
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "earth2studio": earth2studio.__version__,
    "python": platform.python_version(),
}
environment

## Official example

The deterministic workflow from the Earth2Studio examples: DLWP from GFS
initial conditions, 20 steps of 6 hours.

In [ ]:
import earth2studio.run as run
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.models.px import DLWP

started = time.perf_counter()
package = DLWP.load_default_package()
model = DLWP.load_model(package)
io = run.deterministic(["2024-01-01"], 20, model, GFS(), ZarrBackend())
example = {
    "model": "DLWP",
    "steps": 20,
    "t2m_shape": list(io["t2m"].shape),
    "seconds": round(time.perf_counter() - started, 1),
}
del model, io
gc.collect()
torch.cuda.empty_cache()
example

## Model table

Each global model runs 12 steps of 6 hours (3 days) from
2024-07-01 06:00 UTC. StormCast runs 12 steps of 1 hour over its own
domain from its example date. The time covers only the model steps, not
the download of initial conditions. Peak memory is the most GPU memory
PyTorch held during the run.

In [ ]:
from earth2studio.data import ARCO, HRRR, fetch_data
from earth2studio.models import px
from earth2studio.utils.time import to_time_array


def index_inputs(variables):
    names = {str(name) for name in variables}
    levels = sorted(
        {
            int(name[1:])
            for name in names
            if name[:1] in "tquv"
            and name[1:].isdigit()
            and all(f"{prefix}{name[1:]}" in names for prefix in "tquv")
        },
        reverse=True,
    )
    return {
        "t2m": "t2m" in names,
        "d2m": "d2m" in names,
        "sp": "sp" in names,
        "wind_10m": {"u10m", "v10m"} <= names,
        "levels_hpa": levels,
    }


def measure(name, data, start, steps):
    record = {"model": name, "start": start.isoformat(), "steps": steps}
    model = None
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    try:
        began = time.perf_counter()
        model_class = getattr(px, name)
        model = model_class.load_model(model_class.load_default_package())
        model = model.to("cuda").eval()
        record["load_seconds"] = round(time.perf_counter() - began, 1)
        coords = model.input_coords()
        record["inputs"] = index_inputs(model.output_coords(coords)["variable"])
        began = time.perf_counter()
        x, x_coords = fetch_data(
            data,
            to_time_array([start]),
            coords["variable"],
            coords["lead_time"],
            device="cuda",
        )
        record["data_seconds"] = round(time.perf_counter() - began, 1)
        torch.cuda.synchronize()
        began = time.perf_counter()
        with torch.inference_mode():
            for step, _ in enumerate(model.create_iterator(x, x_coords)):
                if step >= steps:
                    break
        torch.cuda.synchronize()
        seconds = time.perf_counter() - began
        record.update(
            fits_on_t4=True,
            forecast_seconds=round(seconds, 1),
            seconds_per_step=round(seconds / steps, 2),
        )
    except torch.cuda.OutOfMemoryError:
        record.update(fits_on_t4=False, error="out of GPU memory")
    except Exception as error:
        record.update(fits_on_t4=None, error=f"{type(error).__name__}: {error}"[:500])
    record["peak_memory_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return record


global_start = datetime(2024, 7, 1, 6)
candidates = [
    ("FCN3", ARCO(), global_start, 12),
    ("SFNO", ARCO(), global_start, 12),
    ("Atlas", ARCO(), global_start, 12),
    ("StormCastCONUS", HRRR(), datetime(2022, 11, 4, 18), 12),
]
records = []
for name, data, start, steps in candidates:
    records.append(measure(name, data, start, steps))
    print(json.dumps(records[-1]))

In [ ]:
result = {"environment": environment, "official_example": example, "models": records}
output = Path("/kaggle/working/earth2_models_t4.json")
output.write_text(json.dumps(result, indent=2))
print(output.read_text())